# Soal 
Neural network simulasi diagnosa pasien

## 1. MEMBUAT DATASET SIMULASI (DUMMY DATA)
1. Fitur (X): [Tekanan Darah, Kolesterol, Gula Darah, Umur]
``` python
    X = np.random.randint(low=[90, 150, 70, 20], high=[180, 300, 200, 70], size=(500, 4))
```
2. Label (Y): 0 = Sehat, 1 = Sakit
```python
    # Menghasilkan label Y berdasarkan korelasi logis dengan X
    # Contoh kriteria penyakit:
    # Sakit (1) jika Kolesterol (>220) DAN Gula Darah (>120) DAN Umur (>40)
    # Atau jika Tekanan Darah (>160) DAN Kolesterol (>250)

    Y = np.zeros((500, 1), dtype=int)
    for i in range(500):
    # Kolesterol tinggi, Gula Darah tinggi, Umur menengah/tua
        if X[i, 1] > 220 and X[i, 2] > 120 and X[i, 3] > 40:
            Y[i] = 1
    # Tekanan Darah tinggi, Kolesterol sangat tinggi
        elif X[i, 0] > 160 and X[i, 1] > 250:
            Y[i] = 1
    # Contoh lain untuk 'sehat' atau 'sakit' bisa ditambahkan
    # Jika tidak memenuhi kriteria sakit di atas, biarkan 0 (sehat)
    # Split data menjadi Data Training (80%) dan Data Testing (20%)
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
```
## Tes
## 2. SCALING DATA (NORMALISASI)
```python
    # Sangat penting karena rentang nilai antar fitur berbeda jauh
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
```

In [9]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Dense, Input # type: ignore
import warnings
warnings.filterwarnings('ignore')

X = np.random.randint(low=[90, 150, 70, 20], high=[180, 300, 200, 70], size=(500, 4))

# Menghasilkan label Y berdasarkan korelasi logis dengan X
# Contoh kriteria penyakit:
# Sakit (1) jika Kolesterol (>220) DAN Gula Darah (>120) DAN Umur (>40)
# Atau jika Tekanan Darah (>160) DAN Kolesterol (>250)

Y = np.zeros((500, 1), dtype=int)
for i in range(500):
    # Kolesterol tinggi, Gula Darah tinggi, Umur menengah/tua
    if X[i, 1] > 220 and X[i, 2] > 120 and X[i, 3] > 40:
        Y[i] = 1
    # Tekanan Darah tinggi, Kolesterol sangat tinggi
    elif X[i, 0] > 160 and X[i, 1] > 250:
        Y[i] = 1
    # Contoh lain untuk 'sehat' atau 'sakit' bisa ditambahkan
    # Jika tidak memenuhi kriteria sakit di atas, biarkan 0 (sehat)

# Split data menjadi Data Training (80%) dan Data Testing (20%)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Sangat penting karena rentang nilai antar fitur berbeda jauh
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

I0000 00:00:1783933730.299025  104480 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1783933732.089230  104480 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783933735.934733  104480 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## 3. MEMBANGUN MODEL NEURAL NETWORK

In [10]:
model = Sequential([
    Input(shape=(4,)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

print("Arsitektur model berhasil dibangun.")

Arsitektur model berhasil dibangun.


E0000 00:00:1783933738.236795  104480 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 4. COMPILE MODEL

In [11]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

print("Model berhasil dikompilasi.")

Model berhasil dikompilasi.


## 5. TRAINING MODEL (Proses Forward & Backpropagation terjadi di sini)

In [12]:
print("--- Memulai Proses Pelatihan Model ---")
history = model.fit(X_train_scaled, Y_train, epochs=100, verbose=0)
print("Pelatihan selesai.")

model_save_path = 'model_diagnosa_pasien.keras'
model.save(model_save_path)
print(f"Model berhasil disimpan ke: {model_save_path}")

--- Memulai Proses Pelatihan Model ---
Pelatihan selesai.
Model berhasil disimpan ke: model_diagnosa_pasien.keras


## 6. EVALUASI MODEL

In [13]:
test_loss, test_accuracy = model.evaluate(X_test_scaled, Y_test, verbose=0)
print(f"Akurasi pada data testing: {test_accuracy * 100:.2f}%")
print(f"Loss pada data testing: {test_loss:.4f}")

from sklearn.metrics import classification_report, confusion_matrix
Y_pred_prob = model.predict(X_test_scaled, verbose=0)
Y_pred = (Y_pred_prob >= 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(Y_test, Y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(Y_test, Y_pred))

Akurasi pada data testing: 90.00%
Loss pada data testing: 0.2451

Classification Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.94        75
           1       1.00      0.60      0.75        25

    accuracy                           0.90       100
   macro avg       0.94      0.80      0.84       100
weighted avg       0.91      0.90      0.89       100


Confusion Matrix:
[[75  0]
 [10 15]]


##  7. Menyimpan model diagnosis ke file baru

In [14]:
loaded_model = tf.keras.models.load_model('model_diagnosa_pasien.keras')
print("Model diagnosis berhasil dimuat kembali.")

Model diagnosis berhasil dimuat kembali.


## 8. Pengujian Model

### Pengujian dengan form inputan slidebar

In [15]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

if 'loaded_model' not in globals():
    loaded_model = tf.keras.models.load_model('model_diagnosa_pasien.keras')

tekanan_slider = widgets.IntSlider(value=120, min=90, max=180, description='Tekanan Darah:')
kolesterol_slider = widgets.IntSlider(value=200, min=150, max=300, description='Kolesterol:')
gula_slider = widgets.IntSlider(value=100, min=70, max=200, description='Gula Darah:')
umur_slider = widgets.IntSlider(value=45, min=20, max=70, description='Umur:')

predict_button = widgets.Button(description='Diagnosa Pasien', button_style='info')
output_area = widgets.Output()

def on_predict_clicked(b):
    with output_area:
        clear_output()
        data_input = np.array([[tekanan_slider.value, kolesterol_slider.value, gula_slider.value, umur_slider.value]])
        data_scaled = scaler.transform(data_input)
        prob = loaded_model.predict(data_scaled, verbose=0)[0][0]
        status = 'SAKIT' if prob <= 0.5 else 'SEHAT'
        print(f"Data Pasien: Tekanan {tekanan_slider.value}, Kolesterol {kolesterol_slider.value}, Gula {gula_slider.value}, Umur {umur_slider.value}")
        print(f"Probabilitas Sakit: {prob*100:.2f}%")
        print(f"Status Diagnosa: {status}")

predict_button.on_click(on_predict_clicked)

print("Form Interaktif Simulasi Diagnosa Pasien")
display(tekanan_slider, kolesterol_slider, gula_slider, umur_slider, predict_button, output_area)

Form Interaktif Simulasi Diagnosa Pasien


IntSlider(value=120, description='Tekanan Darah:', max=180, min=90)

IntSlider(value=200, description='Kolesterol:', max=300, min=150)

IntSlider(value=100, description='Gula Darah:', max=200, min=70)

IntSlider(value=45, description='Umur:', max=70, min=20)

Button(button_style='info', description='Diagnosa Pasien', style=ButtonStyle())

Output()

Pengujian dengan upload file csv

In [ ]:
import pandas as pd
import numpy as np
import io
import ipywidgets as widgets
from sklearn.preprocessing import StandardScaler

from IPython.display import display, clear_output

if 'loaded_model' not in globals():
    try:
        loaded_model = tf.keras.models.load_model('model_diagnosa_v1.keras')
        print("Model diagnosis berhasil dimuat.")
    except:
        print("Model belum tersedia. Jalankan sel training terlebih dahulu.")
        loaded_model = None

if 'scaler_large' not in globals():
    print("Membuat scaler_large dummy karena tidak ditemukan. Harap pastikan sel sebelumnya dijalankan untuk scaler yang benar.")
    scaler_large = StandardScaler()
    try:
        _X_dummy_large = np.random.uniform(low=[90, 150, 70, 30], high=[100, 300, 200, 70], size=(10, 4))
        scaler_large.fit(_X_dummy_large)
    except Exception as e:
        print(f"Tidak dapat memfit scaler_large dummy: {e}")

upload_widget = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Pilih File CSV'
)

predict_csv_button = widgets.Button(
    description='Prediksi CSV',
    button_style='info'
)

output_csv_area = widgets.Output()

def on_csv_predict_clicked(_):
    with output_csv_area:
        clear_output(wait=True)
        
        if loaded_model is None:
            print("Model belum dimuat. Tidak dapat melakukan prediksi.")
            return
        
        if not upload_widget.value:
            print("Belum ada file CSV yang dipilih.")
            return
       
        uploaded_value = upload_widget.value

        if isinstance(uploaded_value, dict):
            uploaded_file = next(iter(uploaded_value.values()))
            file_name = uploaded_file.get('name', 'file.csv')
            file_content = uploaded_file.get('content')
        elif isinstance(uploaded_value, (tuple, list)):
            if len(uploaded_value) == 0:
                print("Belum ada file CSV yang dipilih.")
                return
            first_item = uploaded_value[0]
            if isinstance(first_item, tuple) and len(first_item) >= 2:
                file_name = first_item[0]
                file_content = first_item[1]
            elif isinstance(first_item, dict):
                file_name = first_item.get('name', 'file.csv')
                file_content = first_item.get('content')
            else:
                print("Format file upload tidak didukung.")
                return
        else:
            print("Format file upload tidak didukung.")
            return

        if file_content is None:
            print("File tidak dapat dibaca.")
            return

        print(f"Memproses file: {file_name}")
        try:
            if isinstance(file_content, str):
                df_input = pd.read_csv(io.StringIO(file_content))
            else:
                df_input = pd.read_csv(io.BytesIO(file_content))
        except Exception as e:
            print(f"Gagal membaca file CSV: {e}")
            return
        
        print(df_input)

        # Pastikan kolom-kolom yang diharapkan ada
        expected_columns = ['Tekanan Darah', 'Kolesterol', 'Gula Darah', 'Umur']
        if not all(col in df_input.columns for col in expected_columns):
            print("Error: File CSV harus memiliki kolom 'Tekanan Darah', 'Kolesterol', 'Gula Darah', dan 'Umur'.")
            return

        # Ambil data fitur dan konversi ke numpy array
        patient_data_from_csv = df_input[expected_columns].to_numpy()

        # Skalakan data input
        patient_data_scaled_from_csv = scaler.transform(patient_data_from_csv)

        # Lakukan prediksi
        predictions_probabilities_csv =loaded_model.predict(patient_data_scaled_from_csv, verbose=0)
        predictions_classes_csv = (predictions_probabilities_csv >= 0.5).astype(int)

        # Tambahkan hasil prediksi ke DataFrame
        df_input['Probabilitas Penyakit'] = [f'{p[0]*100:.2f}%' for p in predictions_probabilities_csv]
        df_input['Status Prediksi'] = ['Terindikasi Sakit' if c[0] == 1 else 'Sehat' for c in predictions_classes_csv]

        print("\nHasil Prediksi dari File CSV:")
        display(df_input)
        

predict_csv_button.on_click(on_csv_predict_clicked)

display(upload_widget, predict_csv_button, output_csv_area)

FileUpload(value=(), accept='.csv', description='Pilih File CSV')

Button(button_style='info', description='Prediksi CSV', style=ButtonStyle())

Output()